# BiLSTM sur AG News

Ce notebook entraîne un modèle Bidirectional LSTM sur `train.csv` et `test.csv`.

- `train.csv` est découpé en `train` et `validation`
- `test.csv` reste totalement à part jusqu'à l'évaluation finale
- Optuna optimise `val_accuracy`
- Les métriques suivies restent `accuracy` et `loss`

In [1]:
%pip install --quiet tensorflow scikit-learn matplotlib optuna pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

import optuna

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path(r"D:\100DaysofML\Notebooks\091_Bidirectional_LSTM\archive")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

MAX_VOCAB = 20000
MAX_LEN = 120
BATCH_SIZE = 128
OPTUNA_TRIALS = 5
OPTUNA_EPOCHS = 3
FINAL_EPOCHS = 12

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]


Data directory: D:\100DaysofML\Notebooks\091_Bidirectional_LSTM\archive


In [3]:
def load_news_csv(path):
    df = pd.read_csv(path)
    df = df[["Class Index", "Title", "Description"]].copy()
    df["text"] = (df["Title"].fillna("") + " " + df["Description"].fillna("")).str.strip()
    df["label"] = df["Class Index"].astype(int) - 1
    return df[["text", "label"]]

train_df_full = load_news_csv(TRAIN_PATH)
test_df = load_news_csv(TEST_PATH)

train_df, val_df = train_test_split(
    train_df_full,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df_full["label"],
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorizer.adapt(train_df["text"].values)

X_train = vectorizer(train_df["text"].values)
X_val = vectorizer(val_df["text"].values)
X_test = vectorizer(test_df["text"].values)

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

Train size: 96000
Val size: 24000
Test size: 7600
X_train shape: (96000, 120)
X_val shape: (24000, 120)
X_test shape: (7600, 120)


In [4]:
def build_bilstm_model(embedding_dim=128, lstm_units=64, dense_units=64, dropout=0.3, learning_rate=1e-3):
    inputs = keras.Input(shape=(MAX_LEN,), dtype=tf.int64)
    x = layers.Embedding(MAX_VOCAB, embedding_dim)(inputs)
    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False, dropout=dropout))(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="bilstm_news")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def objective(trial):
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 96, 128])
    lstm_units = trial.suggest_categorical("lstm_units", [32, 64, 96])
    dense_units = trial.suggest_categorical("dense_units", [32, 64])
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)

    model = build_bilstm_model(
        embedding_dim=embedding_dim,
        lstm_units=lstm_units,
        dense_units=dense_units,
        dropout=dropout,
        learning_rate=learning_rate,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=OPTUNA_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
    )

    return max(history.history.get("val_accuracy", [0.0]))

use_optuna = True
if use_optuna:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params
else:
    best_params = {
        "embedding_dim": 128,
        "lstm_units": 64,
        "dense_units": 64,
        "dropout": 0.3,
        "learning_rate": 1e-3,
    }

print("Best params:", best_params)

[I 2026-05-19 00:32:09,661] A new study created in memory with name: no-name-42bcbaac-5696-4417-adff-ae5d5aa6e019


[W 2026-05-19 00:48:45,631] Trial 0 failed with parameters: {'embedding_dim': 64, 'lstm_units': 96, 'dense_units': 32, 'dropout': 0.22377689714489093, 'learning_rate': 0.0003366824240271094} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\afagn\AppData\Local\Temp\ipykernel_21420\4021078922.py", line 37, in objective
    history = model.fit(
              ^^^^^^^^^^
  File "C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\utils\traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\afagn\AppData\Loca

KeyboardInterrupt: 

In [ ]:
final_model = build_bilstm_model(**best_params)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=callbacks,
)

train_loss, train_accuracy = final_model.evaluate(X_train, y_train, verbose=0)
val_loss, val_accuracy = final_model.evaluate(X_val, y_val, verbose=0)
test_loss, test_accuracy = final_model.evaluate(X_test, y_test, verbose=0)

print(f"Train loss: {train_loss:.4f}, Train accuracy: {train_accuracy:.4f}")
print(f"Val loss:   {val_loss:.4f}, Val accuracy:   {val_accuracy:.4f}")
print(f"Test loss:  {test_loss:.4f}, Test accuracy:  {test_accuracy:.4f}")

y_prob = final_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix - BiLSTM")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history.get("accuracy", []), label="train_accuracy")
plt.plot(history.history.get("val_accuracy", []), label="val_accuracy")
plt.axhline(test_accuracy, color="red", linestyle="--", label="test_accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history.get("loss", []), label="train_loss")
plt.plot(history.history.get("val_loss", []), label="val_loss")
plt.axhline(test_loss, color="red", linestyle="--", label="test_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
labels = ["train_acc", "val_acc", "test_acc"]
values = [train_accuracy, val_accuracy, test_accuracy]
plt.bar(labels, values, color=["#4c78a8", "#f58518", "#54a24b"])
plt.ylim(0, 1)
plt.title("Accuracy summary")
plt.tight_layout()
plt.show()